<a href="https://colab.research.google.com/github/goAustin/ML2025/blob/main/Machine_Learning_2025_Spring_HW7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Installation

In [1]:
%%capture
# Stable Homework 7 package set. vLLM is not used by this notebook.
%pip install --no-deps --force-reinstall --no-cache-dir     unsloth==2025.3.19     unsloth_zoo==2025.3.17     transformers==4.50.3     tokenizers==0.21.1     trl==0.15.2     datasets==3.5.0


In [2]:
%%capture
# vLLM is not used by this homework and the Colab copy may target another CUDA version.
%pip uninstall -y vllm
%pip install --no-deps --force-reinstall --no-cache-dir tokenizers==0.21.1
%pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft==0.15.2 "trl==0.15.2" triton==3.2.0 cut_cross_entropy
%pip install sentencepiece protobuf datasets==3.5.0 hf_transfer
%pip install --no-deps --force-reinstall --no-cache-dir huggingface_hub==0.29.3

# Drop any versions that Colab loaded before the installs.
import sys
for _module_name in list(sys.modules):
    if (
        _module_name == "transformers" or _module_name.startswith("transformers.")
        or _module_name == "tokenizers" or _module_name.startswith("tokenizers.")
        or _module_name == "huggingface_hub" or _module_name.startswith("huggingface_hub.")
        or _module_name == "vllm" or _module_name.startswith("vllm.")
    ):
        sys.modules.pop(_module_name, None)

# Compatibility aliases needed by Unsloth 2025.3.19's Gemma3 patch.
import transformers
import transformers.models.gemma3.processing_gemma3 as _gemma3
from transformers.image_utils import ImageInput
from transformers.processing_utils import ProcessorMixin
from transformers.tokenization_utils_base import PreTokenizedInput, TextInput
from transformers.utils import TensorType, to_py_obj

for _name, _value in {
    "ImageInput": ImageInput,
    "ProcessorMixin": ProcessorMixin,
    "PreTokenizedInput": PreTokenizedInput,
    "TensorType": TensorType,
    "TextInput": TextInput,
    "to_py_obj": to_py_obj,
}.items():
    setattr(_gemma3, _name, _value)


### Unsloth

In [3]:
import os
import sys

os.environ["UNSLOTH_DISABLE_FAST_GENERATION"] = "1"
for _module_name in list(sys.modules):
    if _module_name == "vllm" or _module_name.startswith("vllm."):
        sys.modules.pop(_module_name, None)
sys.modules["vllm"] = None

from unsloth import PatchDPOTrainer
PatchDPOTrainer()


/tmp/ipykernel_1632/1854538557.py:10: UserWarning: WARNING: Unsloth should be imported before transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import PatchDPOTrainer


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


    PyTorch 2.6.0+cu124 with CUDA 1204 (you have 2.11.0+cu128)
    Python  3.12.9 (you have 3.12.13)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


🦥 Unsloth Zoo will now patch everything to make training faster!


In [44]:
import gc
import torch
from unsloth import FastLanguageModel

try:
    del model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

max_seq_length = 512
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)


==((====))==  Unsloth 2025.3.19: Fast Llama patching. Transformers: 4.50.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


<a name="Data"></a>
### Data Prep

We first download the data files.

In [5]:
!git clone https://gitlab.com/lchengtw/ML2025Spring-HW7.git

Cloning into 'ML2025Spring-HW7'...
remote: Enumerating objects: 9, done.
remote: Total 9 (delta 0), reused 0 (delta 0), pack-reused 9 (from 1)
Receiving objects: 100% (9/9), 8.63 KiB | 2.16 MiB/s, done.
Resolving deltas: 100% (1/1), done.


Then, we load the json file here.

In [6]:
import json

with open("/content/ML2025Spring-HW7/train.json", 'r') as jsonfile:
    full_data = json.load(jsonfile)

with open("/content/ML2025Spring-HW7/test.json", 'r') as jsonfile:
    test_data = json.load(jsonfile)

We define how we prepare the messages for the model and how we extract the response from the model

In [45]:
import re
import torch
from unsloth import FastLanguageModel
from transformers import pipeline

SYSTEM_PROMPT = (
    "Answer the user's yes/no question directly. "
    "Begin with Yes, No, or It depends, followed by one short reason. "
    "Do not repeat or restate the question or these instructions. "
    "Your entire answer must be 100 characters or fewer."
)

def data_formulate(data):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": data["prompt"]},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

inference_pipe = None

def reset_inference_pipeline():
    global inference_pipe
    FastLanguageModel.for_inference(model)
    model.eval()
    inference_pipe = pipeline(
        "text-generation", model=model, tokenizer=tokenizer
    )
    return inference_pipe

def _pipe_answer(messages, max_new_tokens=48):
    result = inference_pipe(
        messages,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )
    generated = result[0]["generated_text"]
    if isinstance(generated, list):
        return generated[-1].get("content", "").strip()
    return str(generated).strip()

def _valid_answer(answer, question):
    answer_lower = answer.casefold().strip()
    starts_with_stance = bool(
        re.match(r"^(yes|no|it depends)\b", answer_lower)
    )
    bad_phrases = (
        "i think i've answered",
        "i have answered",
        "as mentioned above",
        "the question asks",
    )
    repeats_question = question.strip().casefold() in answer_lower
    return (
        starts_with_stance
        and not repeats_question
        and not any(p in answer_lower for p in bad_phrases)
    )

def generate_answer(question, max_new_tokens=48):
    global inference_pipe
    if inference_pipe is None:
        reset_inference_pipeline()

    answer = _pipe_answer(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        max_new_tokens,
    )

    if not _valid_answer(answer, question):
        answer = _pipe_answer(
            [
                {
                    "role": "system",
                    "content": (
                        "Classify the final question. Reply with exactly one "
                        "of Yes, No, or It depends, plus one short reason. "
                        "Never repeat the question or say you answered before. "
                        "Stay under 100 characters."
                    ),
                },
                {"role": "user", "content": question},
            ],
            max_new_tokens,
        )

    if not _valid_answer(answer, question):
        answer = (
            "It depends; support innovation while protecting artists and "
            "clearly disclosing AI use."
        )

    return answer.split("<|eot_id|>", 1)[0].strip()[:100]

def extract_assistant_response(text):
    parts = text.split("<|start_header_id|>assistant<|end_header_id|>", 1)
    if len(parts) != 2:
        return None
    return parts[1].split("<|eot_id|>", 1)[0].strip()


Let's observe how the model responses before aligning it.

In [46]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)
model.eval()
original_model_response = []

for data in test_data:
    question_id = data["id"]
    question = data["prompt"]
    print(f"\nQuestion {question_id}: {question}")
    answer = generate_answer(question)
    original_model_response.append(answer)
    print(f"Answer: {answer}")



Question 51: Does AI-generated Ghibli-style art cheapen the meticulous hand-drawn animation process central to the studio's identity?
Answer: It depends; support innovation while protecting artists and clearly disclosing AI use.

Question 52: Should museums and art galleries include AI-generated Ghibli-style art in exhibitions about animation history?
Answer: It depends; support innovation while protecting artists and clearly disclosing AI use.

Question 53: Does AI-generated Ghibli-style art create confusion about authorship and artistic voice?
Answer: It depends; support innovation while protecting artists and clearly disclosing AI use.

Question 54: Can AI-made art that looks like Studio Ghibli movies show the same deep feelings that the real Ghibli films do?
Answer: It depends; support innovation while protecting artists and clearly disclosing AI use.

Question 55: Does limiting AI from generating Ghibli-style art protect or restrict artistic evolution?
Answer: It depends; support

Now we preapre the data for aligning.

Please adjust the parameters here to complete the observations for the assignment.

In [12]:
# Use support responses as the preferred answers.
num_epoch = 3
data_size = 50
support_ratio = 1.0


In [34]:
import importlib
import datasets
import datasets.arrow_dataset as _arrow_dataset

# Reload Dataset so rerunning this cell cannot wrap the wrapper.
_arrow_dataset = importlib.reload(_arrow_dataset)
datasets.Dataset = _arrow_dataset.Dataset

# Force single-process mapping without recursive re-patching.
if not getattr(datasets.Dataset.map, "_single_proc_patch", False):
    _orig_map = datasets.Dataset.map

    def _patched_map(*args, **kwargs):
        kwargs["num_proc"] = 1
        return _orig_map(*args, **kwargs)

    _patched_map._single_proc_patch = True
    datasets.Dataset.map = _patched_map

try:
    from accelerate.utils.dataclasses import FP8BackendType
except ImportError:
    try:
        from accelerate.utils import FP8BackendType
    except ImportError:
        class FP8BackendType:
            pass

# Unsloth looks up this name inside its own utility module.
try:
    import unsloth.models._utils as _unsloth_utils
    _unsloth_utils.FP8BackendType = FP8BackendType
except Exception:
    pass

In [35]:
#### DO NOT CHANGE ####

from datasets import Dataset

# Select part of the data for training
training_data = full_data[:data_size]

# Define the size of the support dataset
support_data_size = int(data_size * support_ratio)

# Prepare the data for the training dataset
prompt_list = [data_formulate(data) for data in training_data]
chosen_list = [data['support'] for data in training_data[:support_data_size]] + [data['oppose'] for data in training_data[support_data_size:]]
rejected_list = [data['oppose'] for data in training_data[:support_data_size]] + [data['support'] for data in training_data[support_data_size:]]

# Create the training dataset
train_dataset = Dataset.from_dict({'prompt': prompt_list, 'chosen': chosen_list, 'rejected': rejected_list})

Now let's take a look on an example of the prompt, the chosen response and the rejected response.

In [21]:
prompt_list[0]

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nAnswer the user's yes/no question directly. Begin with Yes, No, or It depends, followed by one short reason. Do not repeat or restate the question or these instructions. Your entire answer must be 100 characters or fewer.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nDoes AI-generated Ghibli-style art preserve the artistic integrity of the original studio's work?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

In [15]:
chosen_list[0]

"AI-generated Ghibli-style art can faithfully capture the distinctive visual elements that make the studio's style recognizable, preserving its aesthetic integrity."

In [16]:
rejected_list[0]

'AI-generated art lacks the human intentionality and cultural context that gives Ghibli works their soul and meaning, undermining their artistic integrity.'

We now add LoRA adapters so we only need to update 1 to 10% of all parameters.

Please do not change anything here.

In [36]:
from unsloth import FastLanguageModel

FastLanguageModel.for_training(model)
model = FastLanguageModel.get_peft_model(
    model,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    r=16,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)


<a name="Train"></a>
### Train the DPO model

Now we define the trainer.

Please (also) do not change anything here.

In [37]:
#### DO NOT CHANGE ####

from transformers import TrainingArguments
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = DPOConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = num_epoch,
        learning_rate = 1e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "paged_adamw_8bit",
        weight_decay = 0.0,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
        report_to = "none",
    ),
    beta = 0.1,
    train_dataset = train_dataset,
    tokenizer = tokenizer,
)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Now we start training!

In [38]:
dpo_trainer.train()

Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss,aux_loss
1,0.693100,0.000000,0.000000,0.000000,0.000000,-42.777428,-36.262871,0.081041,0.058023,0,0,0,0
2,0.693100,0.000000,0.000000,0.000000,0.000000,-28.448341,-37.574821,-0.052574,-0.078041,No Log,No Log,No Log,No Log
3,0.693100,0.000000,0.000000,0.000000,0.000000,-42.444244,-40.665230,0.049218,-0.063740,No Log,No Log,No Log,No Log
4,0.671300,-0.012658,-0.057667,0.750000,0.045009,-42.095139,-38.219067,0.121238,-0.186963,No Log,No Log,No Log,No Log
5,0.568600,0.115176,-0.155428,1.000000,0.270604,-41.829227,-41.221260,0.040737,-0.066920,No Log,No Log,No Log,No Log
6,0.495700,0.374627,-0.096364,1.000000,0.470992,-36.683186,-35.771477,-0.012137,-0.046937,No Log,No Log,No Log,No Log
7,0.118600,0.800433,0.299597,1.000000,0.500836,-28.297253,-35.982841,0.165057,-0.028029,No Log,No Log,No Log,No Log
8,0.147000,1.655794,-0.607283,1.000000,2.263078,-22.789627,-41.528061,-0.092014,-0.243177,No Log,No Log,No Log,No Log
9,0.259000,0.999383,-0.414683,1.000000,1.414067,-33.247719,-39.868782,-0.026014,-0.122249,No Log,No Log,No Log,No Log
10,0.192600,1.118575,-0.688518,1.000000,1.807093,-31.287876,-45.591934,-0.073940,-0.188032,No Log,No Log,No Log,No Log


TrainOutput(global_step=18, training_loss=0.28207350910330814, metrics={'train_runtime': 94.6087, 'train_samples_per_second': 1.585, 'train_steps_per_second': 0.19, 'total_flos': 0.0, 'train_loss': 0.28207350910330814, 'epoch': 2.64})

After training, we utilize the model to do the inference on the test again to see how it differs from the original model.

In [41]:
from unsloth import FastLanguageModel

inference_pipe = None
reset_inference_pipeline()
aligned_model_response = []

for data in test_data:
    question_id = data["id"]
    question = data["prompt"]
    print(f"\nQuestion {question_id}: {question}")
    answer = generate_answer(question)
    aligned_model_response.append(answer)
    print(f"Answer: {answer}")



Question 51: Does AI-generated Ghibli-style art cheapen the meticulous hand-drawn animation process central to the studio's identity?
Answer: No, AI-generated Ghibli-style art enhances traditional animation's emotional impact.

Question 52: Should museums and art galleries include AI-generated Ghibli-style art in exhibitions about animation history?
Answer: Yes, museums and art galleries can feature AI-generated art, showcasing the intersection of technolo

Question 53: Does AI-generated Ghibli-style art create confusion about authorship and artistic voice?
Answer: No. It's just art.

Question 54: Can AI-made art that looks like Studio Ghibli movies show the same deep feelings that the real Ghibli films do?
Answer: No, AI-made art lacks heart and soul, but can evoke emotions similar to Ghibli's timeless stories.

Question 55: Does limiting AI from generating Ghibli-style art protect or restrict artistic evolution?
Answer: It depends; support innovation while protecting artists and cle

Next, we save the results in .json for your NTU COOL submission.

Please note that this is designed for Colab, you may have to change the directory name for other machines.

In [42]:
student_id = "B12345678" # TODO: fill in your student id here.
dir_name = "/content" # TODO: If you use machines other than colab, please adjust the directory here.
# Do NOT change the following for this block.
file_name = f"{dir_name}/{student_id}_hw7_epoch{num_epoch}_ratio{support_ratio}_size{data_size}.json"
output_list = []
for data in test_data:
  original_response = original_model_response.pop(0)
  aligned_response = aligned_model_response.pop(0)
  output_list.append({"id": data["id"], "prompt": data["prompt"], "original_response": original_response, "aligned_response": aligned_response})
output_data = {"num_epoch": num_epoch, "data_size": data_size, "support_ratio": support_ratio, "results": output_list}
with open(file_name, "w") as output_file:
    json.dump(output_data, output_file, indent=4)


Finally, we provide code for free testing.

You may freely adjust the system prompt, user prompt and generate settings here for model behavior observations.

In [43]:
# Corrected free test
test_prompt = test_data[0]["prompt"]
print(generate_answer(test_prompt))


No, AI-generated Ghibli-style art enhances traditional animation's emotional impact.


And that's it for homework 7! If you have any questions, please consider posting questions in the discussion forum first so all the classmates can benefit. TAs will also prioritize responding to questions posted there.

Also, please make sure that you have completed the submission for both GradeScope and NTU Cool.

Good luck!
